# Notebook 07 — LoRA Hyperparameters and Target Modules

After a first QLoRA run works, the next question is not "which trick is trendy?" It is "which knobs actually change adapter behavior, cost, and stability on Colab-class hardware?" This notebook answers that by running controlled comparisons.

## What you will build

This notebook keeps the dataset, split, decoding prompts, and training budget fixed while changing one adapter decision at a time:

1. rank and alpha
2. dropout
3. selective versus all-linear targeting
4. practical framing for rsLoRA and LoftQ

The goal is not a universal winner. The goal is a reproducible way to choose a default.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Step 1: Define notebook paths and helper functions

We will keep every experiment short, save compact artifacts, and compute the same holdout rubric for each run.

In [ ]:
import gc

random.seed(7)

ARTIFACT_DIR = Path(OUTPUT_DIR) / "notebook_07_lora_hyperparameters"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def render_chat(record, active_tokenizer):
    messages = [
        {"role": "system", "content": "You are Castalia Mentor, a concise tutor focused on practical finetuning guidance."},
        {"role": "user", "content": record["user"]},
        {"role": "assistant", "content": record["assistant"]},
    ]
    return active_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def build_formatted_dataset(rows, active_tokenizer):
    return Dataset.from_list([
        {**row, "text": render_chat(row, active_tokenizer)}
        for row in rows
    ])

def count_parameters(active_model):
    total_params = sum(param.numel() for param in active_model.parameters())
    trainable_params = sum(param.numel() for param in active_model.parameters() if param.requires_grad)
    return total_params, trainable_params

def generate_completion(active_model, active_tokenizer, prompt, max_new_tokens=80):
    messages = [
        {"role": "system", "content": "You are Castalia Mentor, a concise tutor focused on practical finetuning guidance."},
        {"role": "user", "content": prompt},
    ]
    try:
        FastLanguageModel.for_inference(active_model)
    except Exception:
        pass
    input_ids = active_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(active_model.device)
    outputs = active_model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )
    completion_ids = outputs[0][input_ids.shape[-1]:]
    return active_tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

def anchor_score(text, expected_terms):
    normalized = text.lower()
    return round(sum(term in normalized for term in expected_terms) / len(expected_terms), 3)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Artifacts:", ARTIFACT_DIR.resolve())

## Step 2: Create a compact local dataset

The dataset is intentionally narrow. Small, focused datasets make adapter-capacity trade-offs easier to see because the model is not trying to learn ten different things at once.

In [ ]:
examples = [
    {"id": "01", "user": "Why do tiny first finetuning runs matter?", "assistant": "Tiny first runs surface formatting mistakes, unstable settings, and evaluation gaps before you spend hours on a longer job."},
    {"id": "02", "user": "Why is validation still useful on a small dataset?", "assistant": "Validation still matters because lower training loss can mean either better generalization or simple memorization."},
    {"id": "03", "user": "What does LoRA rank control?", "assistant": "LoRA rank controls the width of the low-rank update. Higher rank means more capacity, more trainable parameters, and more overfitting risk."},
    {"id": "04", "user": "How should lora_alpha be interpreted?", "assistant": "lora_alpha scales the adapter update, so it changes the effective strength of the learned low-rank correction."},
    {"id": "05", "user": "Why can dropout help on small finetuning sets?", "assistant": "Dropout regularizes the adapter update, which can reduce brittleness when the dataset is repetitive or narrow."},
    {"id": "06", "user": "Why do teams start with q_proj and v_proj so often?", "assistant": "They are a cheap selective baseline that often changes attention behavior without paying for broad coverage."},
    {"id": "07", "user": "Why expand beyond attention-only targets?", "assistant": "Adding MLP projections increases the update surface, which often helps when attention-only adapters feel too weak."},
    {"id": "08", "user": "Why is all-linear targeting a common QLoRA default?", "assistant": "It covers both attention and MLP projections, so the adapter can express richer updates while staying parameter efficient."},
    {"id": "09", "user": "When is a higher rank wasteful?", "assistant": "It is wasteful when validation quality and useful completions do not improve enough to justify extra parameters and runtime."},
    {"id": "10", "user": "How do you compare LoRA configurations fairly?", "assistant": "Keep the dataset, split, prompts, budget, and decode settings fixed so the adapter choice is the main difference."},
    {"id": "11", "user": "Why inspect runtime as well as loss?", "assistant": "Iteration speed matters on Colab because a slightly better setting can still be a bad default if it slows every experiment."},
    {"id": "12", "user": "Why write target modules explicitly?", "assistant": "Explicit target modules make runs reproducible and prevent silent drift in what was actually adapted."},
]

raw_dataset = Dataset.from_list(examples)
split_dataset = raw_dataset.train_test_split(test_size=0.25, seed=3407)
train_rows = split_dataset["train"].to_list()
eval_rows = split_dataset["test"].to_list()

display(pd.DataFrame(examples)[["id", "user"]])
print("Train rows:", len(train_rows), "Validation rows:", len(eval_rows))

## Step 3: Inspect prompt lengths before training

This is a quick sanity check that the examples are small enough for cheap ablations and that large runtime swings will mostly come from the adapter settings, not sequence-length chaos.

In [ ]:
length_df = pd.DataFrame(examples)
length_df["text"] = length_df.apply(lambda row: render_chat(row, tokenizer), axis=1)
length_df["token_length"] = length_df["text"].map(lambda text: len(tokenizer(text, add_special_tokens=False)["input_ids"]))

display(length_df[["id", "token_length"]].sort_values("token_length"))
print("Mean token length:", round(length_df["token_length"].mean(), 1))
print("Max token length:", int(length_df["token_length"].max()))

## Step 4: Define the holdout rubric

We use a tiny, consistent rubric based on anchor terms. This is not a final benchmark. It is just enough structure to compare small runs on equal footing.

In [ ]:
holdout_prompts = [
    {
        "prompt_id": "rank_tradeoff",
        "prompt": "Explain the main trade-off of raising LoRA rank.",
        "expected_terms": ["capacity", "parameters", "overfitting"],
    },
    {
        "prompt_id": "targeting_tradeoff",
        "prompt": "Why can all-linear targeting outperform a very selective target set?",
        "expected_terms": ["mlp", "coverage", "capacity"],
    },
    {
        "prompt_id": "fair_comparison",
        "prompt": "How do you run a fair LoRA ablation on Colab?",
        "expected_terms": ["fixed", "dataset", "budget"],
    },
]

pd.DataFrame(holdout_prompts)

## Step 5: Frame the adapter knobs from first principles

Before we train anything, state what each knob is expected to change. That keeps later interpretations grounded in mechanism instead of folklore.

In [ ]:
knob_df = pd.DataFrame(
    [
        {"knob": "rank", "changes": "adapter capacity", "practical_cost": "more trainable parameters and slightly more runtime"},
        {"knob": "alpha", "changes": "update scale", "practical_cost": "can sharpen learning or make comparisons misleading if rank changes too"},
        {"knob": "dropout", "changes": "regularization", "practical_cost": "may stabilize tiny datasets but can slow fitting"},
        {"knob": "target_modules", "changes": "where updates apply", "practical_cost": "broader coverage increases expressiveness and parameter count"},
    ]
)
display(knob_df)

## Step 6: Define a controlled experiment grid

Each configuration isolates one lesson:

- selective low-capacity baseline
- rank and alpha increase with the same target modules
- dropout change with the same targets and scale
- all-linear coverage at the same nominal rank
- a higher-capacity all-linear follow-up

In [ ]:
experiment_configs = [
    {
        "name": "selective_r8_alpha16",
        "family": "selective",
        "r": 8,
        "alpha": 16,
        "dropout": 0.0,
        "target_modules": ["q_proj", "v_proj"],
        "note": "cheap selective baseline",
    },
    {
        "name": "selective_r16_alpha32",
        "family": "selective",
        "r": 16,
        "alpha": 32,
        "dropout": 0.0,
        "target_modules": ["q_proj", "v_proj"],
        "note": "same targets, higher capacity and scale",
    },
    {
        "name": "selective_r16_alpha32_dropout",
        "family": "selective",
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "target_modules": ["q_proj", "v_proj"],
        "note": "same selective setup, regularized",
    },
    {
        "name": "all_linear_r16_alpha32",
        "family": "all_linear",
        "r": 16,
        "alpha": 32,
        "dropout": 0.0,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "note": "practical QLoRA-style default",
    },
    {
        "name": "all_linear_r32_alpha64_dropout",
        "family": "all_linear",
        "r": 32,
        "alpha": 64,
        "dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "note": "higher-capacity follow-up",
    },
]

config_df = pd.DataFrame(experiment_configs)
config_df["alpha_over_r"] = (config_df["alpha"] / config_df["r"]).round(2)
config_df["target_count"] = config_df["target_modules"].map(len)
display(config_df[["name", "family", "r", "alpha", "alpha_over_r", "dropout", "target_count", "note"]])

## Step 7: Add an expected-comparison map

This is the real control surface for the notebook. We will only make claims that line up with a direct like-for-like comparison.

In [ ]:
comparison_plan_df = pd.DataFrame(
    [
        {"comparison": "rank_and_alpha", "left": "selective_r8_alpha16", "right": "selective_r16_alpha32", "what_is_held_constant": "dataset, targets, budget, dropout"},
        {"comparison": "dropout", "left": "selective_r16_alpha32", "right": "selective_r16_alpha32_dropout", "what_is_held_constant": "dataset, targets, budget, rank, alpha"},
        {"comparison": "target_modules", "left": "selective_r16_alpha32", "right": "all_linear_r16_alpha32", "what_is_held_constant": "dataset, budget, rank, alpha, dropout"},
        {"comparison": "scale_up", "left": "all_linear_r16_alpha32", "right": "all_linear_r32_alpha64_dropout", "what_is_held_constant": "dataset, family, broad coverage"},
    ]
)
display(comparison_plan_df)

## Step 8: Release the setup model before the ablations

The shared setup cell loads one adapterized model for convenience. We no longer need that object because each ablation reloads a fresh base model with its own configuration.

In [ ]:
try:
    del model
except NameError:
    pass

clear_gpu()
print("Released the setup model. The tokenizer remains available for dataset inspection.")

## Step 9: Define the short experiment runner

Every run will:

1. load the same base model
2. attach the specified LoRA configuration
3. train for the same short budget
4. score the same holdout prompts
5. save a summary for later inspection

In [ ]:
def run_experiment(config):
    experiment_dir = ARTIFACT_DIR / config["name"]
    experiment_dir.mkdir(parents=True, exist_ok=True)
    start_time = time.time()

    exp_model, exp_tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
        cache_dir=CACHE_DIR,
    )
    exp_tokenizer.padding_side = "right"

    exp_model = FastLanguageModel.get_peft_model(
        exp_model,
        r=config["r"],
        lora_alpha=config["alpha"],
        lora_dropout=config["dropout"],
        bias="none",
        target_modules=config["target_modules"],
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

    exp_train_dataset = build_formatted_dataset(train_rows, exp_tokenizer)
    exp_eval_dataset = build_formatted_dataset(eval_rows, exp_tokenizer)

    training_args = TrainingArguments(
        output_dir=str(experiment_dir),
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        max_steps=12,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        evaluation_strategy="steps",
        eval_steps=6,
        save_steps=12,
        save_total_limit=1,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        report_to="none",
        seed=3407,
    )

    trainer = SFTTrainer(
        model=exp_model,
        tokenizer=exp_tokenizer,
        train_dataset=exp_train_dataset,
        eval_dataset=exp_eval_dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        args=training_args,
    )

    trainer.train()
    log_df = pd.DataFrame(trainer.state.log_history)

    prompt_rows = []
    for item in holdout_prompts:
        completion = generate_completion(exp_model, exp_tokenizer, item["prompt"])
        prompt_rows.append(
            {
                "prompt_id": item["prompt_id"],
                "completion": completion,
                "anchor_score": anchor_score(completion, item["expected_terms"]),
            }
        )

    prompt_df = pd.DataFrame(prompt_rows)
    total_params, trainable_params = count_parameters(exp_model)
    duration_s = round(time.time() - start_time, 2)

    summary = {
        "name": config["name"],
        "family": config["family"],
        "r": config["r"],
        "alpha": config["alpha"],
        "alpha_over_r": round(config["alpha"] / config["r"], 2),
        "dropout": config["dropout"],
        "target_count": len(config["target_modules"]),
        "targets": ", ".join(config["target_modules"]),
        "trainable_params": int(trainable_params),
        "trainable_pct": round(100 * trainable_params / total_params, 4),
        "final_train_loss": round(float(log_df["loss"].dropna().iloc[-1]), 4),
        "final_eval_loss": round(float(log_df["eval_loss"].dropna().iloc[-1]), 4) if "eval_loss" in log_df and not log_df["eval_loss"].dropna().empty else None,
        "mean_anchor_score": round(float(prompt_df["anchor_score"].mean()), 3),
        "duration_s": duration_s,
        "sample_completion": prompt_df.iloc[0]["completion"],
        "note": config["note"],
    }

    with open(experiment_dir / "summary.json", "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)
    prompt_df.to_csv(experiment_dir / "holdout_predictions.csv", index=False)

    del trainer
    del exp_model
    del exp_tokenizer
    clear_gpu()

    return summary

## Step 10: Run the controlled comparisons

This is the expensive cell. On Colab Pro, keep the budget fixed and short so the ablation remains interpretable and affordable.

In [ ]:
experiment_results = [run_experiment(config) for config in experiment_configs]
results_df = pd.DataFrame(experiment_results).sort_values(["mean_anchor_score", "final_eval_loss"], ascending=[False, True]).reset_index(drop=True)
results_df.to_csv(ARTIFACT_DIR / "experiment_results.csv", index=False)
display(results_df)

## Step 11: Look at the overall leaderboard

The main table should combine quality, parameter count, and wall-clock cost. That is the real decision surface for Colab-scale work.

In [ ]:
leaderboard_df = results_df[
    ["name", "family", "r", "alpha", "dropout", "trainable_params", "final_eval_loss", "mean_anchor_score", "duration_s"]
].copy()
display(leaderboard_df)

## Step 12: Compare rank and alpha while targets stay fixed

This is the cleanest comparison for capacity. The only meaningful change between these runs is that the low-rank update becomes wider and more strongly scaled.

In [ ]:
rank_alpha_df = results_df[results_df["name"].isin(["selective_r8_alpha16", "selective_r16_alpha32"])].copy()
rank_alpha_df["delta_vs_baseline_score"] = (rank_alpha_df["mean_anchor_score"] - rank_alpha_df["mean_anchor_score"].iloc[0]).round(3)
rank_alpha_df["delta_vs_baseline_eval_loss"] = (rank_alpha_df["final_eval_loss"] - rank_alpha_df["final_eval_loss"].iloc[0]).round(4)
display(rank_alpha_df[["name", "r", "alpha", "alpha_over_r", "trainable_params", "mean_anchor_score", "delta_vs_baseline_score", "final_eval_loss", "delta_vs_baseline_eval_loss"]])

## Step 13: Compare dropout while everything else stays fixed

Dropout is easiest to reason about when rank, alpha, targets, and budget are unchanged.

In [ ]:
dropout_df = results_df[results_df["name"].isin(["selective_r16_alpha32", "selective_r16_alpha32_dropout"])].copy()
dropout_df["is_dropout_run"] = dropout_df["dropout"] > 0
display(dropout_df[["name", "dropout", "mean_anchor_score", "final_eval_loss", "duration_s", "sample_completion"]])

## Step 14: Compare selective versus all-linear targeting

This is the most important scope decision in practical QLoRA work. Broad coverage often matters as much as rank.

In [ ]:
targeting_df = results_df[results_df["name"].isin(["selective_r16_alpha32", "all_linear_r16_alpha32"])].copy()
targeting_df["coverage_style"] = targeting_df["family"].map({"selective": "q_proj + v_proj only", "all_linear": "attention + MLP projections"})
display(targeting_df[["name", "coverage_style", "target_count", "trainable_params", "mean_anchor_score", "final_eval_loss", "duration_s"]])

## Step 15: Visualize capacity versus outcome

This chart makes it easier to see whether extra trainable parameters bought useful quality or merely extra cost.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

results_df.plot(
    kind="scatter",
    x="trainable_params",
    y="mean_anchor_score",
    s=90,
    color="tab:blue",
    ax=axes[0],
    title="Trainable parameters vs holdout score",
)
for _, row in results_df.iterrows():
    axes[0].annotate(row["name"], (row["trainable_params"], row["mean_anchor_score"]), fontsize=8)

results_df.plot(
    kind="scatter",
    x="duration_s",
    y="final_eval_loss",
    s=90,
    color="tab:orange",
    ax=axes[1],
    title="Runtime vs validation loss",
)
for _, row in results_df.iterrows():
    axes[1].annotate(row["name"], (row["duration_s"], row["final_eval_loss"]), fontsize=8)

plt.tight_layout()

## Step 16: Inspect target-module coverage by family

This turns the target list into a simple explanation of what changed: attention only or attention plus MLP.

In [ ]:
coverage_rows = []
for config in experiment_configs:
    coverage_rows.append(
        {
            "name": config["name"],
            "attention_targets": sum(module in {"q_proj", "k_proj", "v_proj", "o_proj"} for module in config["target_modules"]),
            "mlp_targets": sum(module in {"gate_proj", "up_proj", "down_proj"} for module in config["target_modules"]),
            "coverage_style": "all_linear" if "gate_proj" in config["target_modules"] else "selective",
        }
    )

coverage_df = pd.DataFrame(coverage_rows).merge(
    results_df[["name", "trainable_params", "mean_anchor_score"]],
    on="name",
    how="left",
)
display(coverage_df)

## Step 17: Save a compact comparison summary

Artifacts matter because later notebooks may want to reuse the best default rather than re-reading a chart by eye.

In [ ]:
summary_rows = [
    {
        "comparison": "rank_and_alpha",
        "winner_by_eval_loss": rank_alpha_df.sort_values("final_eval_loss").iloc[0]["name"],
    },
    {
        "comparison": "dropout",
        "winner_by_eval_loss": dropout_df.sort_values("final_eval_loss").iloc[0]["name"],
    },
    {
        "comparison": "target_modules",
        "winner_by_eval_loss": targeting_df.sort_values("final_eval_loss").iloc[0]["name"],
    },
]

comparison_summary_df = pd.DataFrame(summary_rows)
comparison_summary_df.to_csv(ARTIFACT_DIR / "comparison_summary.csv", index=False)
display(comparison_summary_df)

## Step 18: Frame rsLoRA and LoftQ without overpromising

These ideas matter, but they are follow-up methods. They do not replace disciplined baseline comparisons.

In [ ]:
advanced_method_df = pd.DataFrame(
    [
        {
            "method": "rsLoRA",
            "core_idea": "rank-stabilized scaling so higher-rank updates behave more consistently",
            "good_reason_to_try": "you are sweeping rank more aggressively and want cleaner scaling behavior",
            "why_not_the_default_here": "the first lesson is still ordinary rank, alpha, dropout, and target-module control",
        },
        {
            "method": "LoftQ",
            "core_idea": "quantization-aware initialization for LoRA-style training",
            "good_reason_to_try": "quantized runs seem brittle and you want a more advanced initialization study",
            "why_not_the_default_here": "it adds another moving part before the basic adapter-scope story is clear",
        },
    ]
)
display(advanced_method_df)

## Step 19: Write a practical decision rubric

The notebook is only useful if it turns into a default you would actually carry into the next finetuning run.

In [ ]:
decision_rubric_df = pd.DataFrame(
    [
        {"question": "Need the cheapest baseline?", "default_direction": "start selective with lower rank"},
        {"question": "Need stronger task adaptation on QLoRA?", "default_direction": "test all-linear at moderate rank before blindly scaling rank upward"},
        {"question": "Seeing instability on tiny data?", "default_direction": "try modest dropout or lower rank before increasing scope"},
        {"question": "Planning an advanced quantized study?", "default_direction": "consider rsLoRA or LoftQ only after standard comparisons are stable"},
    ]
)
display(decision_rubric_df)

## Step 20: Pick the notebook default

A default should balance quality and iteration speed, not just win one metric.

In [ ]:
default_sort_df = results_df.copy()
default_sort_df["composite_rank"] = default_sort_df["final_eval_loss"].rank(method="dense") + default_sort_df["duration_s"].rank(method="dense")
recommended_default = default_sort_df.sort_values(["composite_rank", "mean_anchor_score"]).iloc[0]

print("Recommended default for the next Colab-sized run:")
print(recommended_default[["name", "family", "r", "alpha", "dropout", "targets", "final_eval_loss", "duration_s"]])

## Step 21: Read one qualitative sample from the recommended run

Quantitative summaries narrow the field. A quick qualitative check confirms the run still sounds like the intended tutor behavior.

In [ ]:
print(recommended_default["sample_completion"])

## Step 22: Record the notebook conclusion as JSON

This makes the recommendation easy to hand off to later notebooks or deployment notes.

In [ ]:
notebook_conclusion = {
    "recommended_default": recommended_default["name"],
    "reason": "best balance of validation loss and short-run cost",
    "all_linear_is_often_strong": bool((results_df["family"] == "all_linear").any()),
    "advanced_methods_framed": ["rsLoRA", "LoftQ"],
}

with open(ARTIFACT_DIR / "notebook_conclusion.json", "w", encoding="utf-8") as handle:
    json.dump(notebook_conclusion, handle, indent=2)

notebook_conclusion

## Key takeaways

- Rank and target modules are usually the first big LoRA levers.
- Alpha only makes sense when interpreted together with rank.
- Dropout is a regularization knob, not an automatic upgrade.
- All-linear targeting is often a strong practical QLoRA default on open instruct models.
- rsLoRA and LoftQ are useful follow-ups after basic controlled comparisons are already understood.